In [1]:
import os

bronze_path = "/lakehouse/default/Files/bronze"

print(os.listdir(bronze_path))

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 3, Finished, Available, Finished, False)

['brazil_oil_production.sqlite']


In [2]:
import pandas as pd

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 4, Finished, Available, Finished, False)

In [3]:
import sqlite3

db_path = "/lakehouse/default/Files/bronze/brazil_oil_production.sqlite"

conn = sqlite3.connect(db_path)

cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

tables = cursor.fetchall()

print(tables)

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 5, Finished, Available, Finished, False)

[('production',)]


In [4]:
import pandas as pd

df = pd.read_sql_query("SELECT * FROM production LIMIT 5", conn)

display(df)

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d77d274f-412e-4017-98fc-7ec902a67fdd)

In [5]:
df = pd.read_sql_query("SELECT * FROM production", conn)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 7, Finished, Available, Finished, False)

Rows: 645989
Columns: 21

Column names:
['Year', 'Month', 'State', 'Basin', 'Field', 'Well', 'Environment', 'Installation', 'Oil (m³)', 'Condensate oil (m³)', 'Associated petroleum gas (Mm³)', 'Non-associated petroleum gas (Mm³)', 'Water (m³)', 'Gas injection (Mm³)', 'Secondary recovery water injection (m³)', 'Wastewater injection (m³)', 'CO2 injection (Mm³)', 'Nitrogen injection (Mm³)', 'Steam injection (t)', 'Polymer injection (m³)', 'Others fluids injection (m³)']

Missing values:
Year                                            0
Month                                           0
State                                           0
Basin                                           0
Field                                           0
Well                                            0
Environment                                     0
Installation                               335701
Oil (m³)                                   181218
Condensate oil (m³)                        627122
Associated 

In [6]:
print(df.dtypes)

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 8, Finished, Available, Finished, False)

Year                                         int64
Month                                        int64
State                                       object
Basin                                       object
Field                                       object
Well                                        object
Environment                                 object
Installation                                object
Oil (m³)                                    object
Condensate oil (m³)                         object
Associated petroleum gas (Mm³)             float64
Non-associated petroleum gas (Mm³)         float64
Water (m³)                                 float64
Gas injection (Mm³)                        float64
Secondary recovery water injection (m³)    float64
Wastewater injection (m³)                  float64
CO2 injection (Mm³)                        float64
Nitrogen injection (Mm³)                   float64
Steam injection (t)                        float64
Polymer injection (m³)         

In [7]:
print(df[['Year', 'Month']].drop_duplicates().sort_values(['Year', 'Month']).head(20))

print("\nYears:")
print(sorted(df['Year'].unique()))

print("\nMonths:")
print(sorted(df['Month'].unique()))

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 9, Finished, Available, Finished, False)

        Year  Month
174967  1954     12
174968  1955      1
174969  1955      2
174970  1955      3
174973  1955      4
174975  1955      5
174976  1955      6
174977  1955      7
174979  1955      8
174980  1955      9
174982  1955     10
174984  1955     11
174987  1955     12
174991  1956      1
174994  1956      2
174997  1956      3
174999  1956      4
175001  1956      5
175004  1956      6
175009  1956      7

Years:
[1954, 1955, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]

Months:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


In [8]:
problem_columns = [
    'Oil (m³)',
    'Condensate oil (m³)',
    'Others fluids injection (m³)'
]

for col in problem_columns:
    numeric = pd.to_numeric(df[col], errors='coerce')

    invalid = df[
        df[col].notna() & numeric.isna()
    ][col].unique()

    print(f"\n{col}")
    print("Invalid values:", invalid[:20])
    print("Count:", len(invalid))

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 10, Finished, Available, Finished, False)


Oil (m³)
Invalid values: [' 28 e 5.']
Count: 1

Condensate oil (m³)
Invalid values: [' 08 e 09']
Count: 1

Others fluids injection (m³)
Invalid values: ['0\r\n2019.07/2019.Alagoas.Alagoas.FURADO.3-FU-4-AL.Terra.Não Informado.0.0.0.0.0........\r\n2019.07/2019.Alagoas.Alagoas.FURADO.3-FU-5-AL.Terra.Estação Coletora de Furado.33'
 '0\r\n2019.07/2019.Alagoas.Alagoas.FURADO.7-FU-17-AL.Terra.Não Informado.0.0.0.0.0........\r\n2019.07/2019.Alagoas.Alagoas.FURADO.7-FU-170D-AL.Terra.Estação Coletora de Furado.248'
 '0\r\n2019.07/2019.Alagoas.Alagoas.FURADO.7-FU-27-AL.Terra.Não Informado.0.0.0.0.0........\r\n2019.07/2019.Alagoas.Alagoas.FURADO.7-FU-28-AL.Terra.Não Informado.0.0.0.0.0........\r\n2019.07/2019.Alagoas.Alagoas.FURADO.7-FU-29-AL.Terra.Estação Coletora de Furado.0.0.0.766'
 '0\r\n2019.07/2019.Alagoas.Alagoas.PILAR.7-PIR-105-AL.Terra.Não Informado.0.0.0.0.0........\r\n2019.07/2019.Alagoas.Alagoas.PILAR.7-PIR-107-AL.Terra.Não Informado.0.0.0.0.0........\r\n2019.07/2019.Alagoas.Alagoas.

In [9]:
bad_others = pd.to_numeric(
    df['Others fluids injection (m³)'],
    errors='coerce'
).isna() & df['Others fluids injection (m³)'].notna()

print("Affected rows:", bad_others.sum())

print("\nYears affected:")
print(df.loc[bad_others, 'Year'].value_counts().sort_index())

print("\nMonths affected:")
print(
    df.loc[bad_others, ['Year', 'Month']]
      .value_counts()
      .sort_index()
)

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 11, Finished, Available, Finished, False)

Affected rows: 12067

Years affected:
Year
2019    6066
2020    6001
Name: count, dtype: int64

Months affected:
Year  Month
2019  7        1025
      8        1020
      9        1001
      10       1021
      11        998
      12       1001
2020  1         992
      2        1009
      3        1012
      7        1000
      8         997
      9         991
Name: count, dtype: int64


In [10]:
numeric_columns = [
    'Associated petroleum gas (Mm³)',
    'Non-associated petroleum gas (Mm³)',
    'Water (m³)',
    'Gas injection (Mm³)',
    'Secondary recovery water injection (m³)',
    'Wastewater injection (m³)',
    'CO2 injection (Mm³)',
    'Nitrogen injection (Mm³)',
    'Steam injection (t)',
    'Polymer injection (m³)'
]

for col in numeric_columns:
    print(f"\n{col}")
    print("Min:", df[col].min())
    print("Max:", df[col].max())
    print("Negative values:", (df[col] < 0).sum())

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 12, Finished, Available, Finished, False)


Associated petroleum gas (Mm³)
Min: 1e-05
Max: 190000.0
Negative values: 0

Non-associated petroleum gas (Mm³)
Min: 0.00142
Max: 78298.0
Negative values: 0

Water (m³)
Min: -0.0268
Max: 258875.63389
Negative values: 3

Gas injection (Mm³)
Min: 0.00023
Max: 106095.354
Negative values: 0

Secondary recovery water injection (m³)
Min: 0.00537
Max: 327904.779
Negative values: 0

Wastewater injection (m³)
Min: 1.0
Max: 134224.0
Negative values: 0

CO2 injection (Mm³)
Min: 0.01166
Max: 67407.0
Negative values: 0

Nitrogen injection (Mm³)
Min: 7452.0
Max: 19038.0
Negative values: 0

Steam injection (t)
Min: 3.0
Max: 6039.0
Negative values: 0

Polymer injection (m³)
Min: 0.02
Max: 999.0
Negative values: 0


In [11]:
negative_water = df[df['Water (m³)'] < 0]

display(
    negative_water[
        ['Year', 'Month', 'State', 'Basin', 'Field', 'Well', 'Water (m³)']
    ]
)

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cbaf0578-7fe8-4209-a311-3d127ba1d57b)

In [12]:
silver_df=df.copy()
print("Row before cleaning : ",len(silver_df))

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 14, Finished, Available, Finished, False)

Row before cleaning :  645989


In [13]:
numeric_fix_columns = [
    'Oil (m³)',
    'Condensate oil (m³)',
    'Others fluids injection (m³)'
]

for col in numeric_fix_columns:
    silver_df[col] = pd.to_numeric(
        silver_df[col],
        errors='coerce'
    )

print(silver_df[numeric_fix_columns].dtypes)

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 15, Finished, Available, Finished, False)

Oil (m³)                        float64
Condensate oil (m³)             float64
Others fluids injection (m³)    float64
dtype: object


In [14]:
silver_df.loc[silver_df['Water (m³)'] < 0, 'Water (m³)'] = pd.NA

print("Negative water values after cleaning:",
      (silver_df['Water (m³)'] < 0).sum())

print("Missing water values after cleaning:",
      silver_df['Water (m³)'].isna().sum())

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 16, Finished, Available, Finished, False)

Negative water values after cleaning: 0
Missing water values after cleaning: 236216


In [15]:
missing_summary = pd.DataFrame({
    'Missing_Count': silver_df.isna().sum(),
    'Missing_Percent': (silver_df.isna().mean() * 100).round(2)
})

missing_summary = missing_summary[
    missing_summary['Missing_Count'] > 0
].sort_values('Missing_Percent', ascending=False)

display(missing_summary)

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cc474aa9-eeda-41fd-b4fd-1ff6ffe34eb0)

In [16]:
missing_summary_display = missing_summary.reset_index()
missing_summary_display.columns = [
    'Column',
    'Missing_Count',
    'Missing_Percent'
]

display(missing_summary_display)

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7eed8d4e-9913-48bb-aa7a-23a4fdeec7d0)

In [17]:
silver_df['Installation'] = silver_df['Installation'].fillna('Not Reported')

print("Missing Installation:",
      silver_df['Installation'].isna().sum())

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 19, Finished, Available, Finished, False)

Missing Installation: 0


In [18]:
silver_df['Installation'] = silver_df['Installation'].fillna('Not Reported')

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 20, Finished, Available, Finished, False)

In [19]:
silver_df['Date'] = pd.to_datetime(
    dict(
        year=silver_df['Year'],
        month=silver_df['Month'],
        day=1
    )
)

print(silver_df[['Year', 'Month', 'Date']].head())

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 21, Finished, Available, Finished, False)

   Year  Month       Date
0  2019      4 2019-04-01
1  2019      4 2019-04-01
2  2019      4 2019-04-01
3  2019      4 2019-04-01
4  2019      4 2019-04-01


In [20]:
silver_df = silver_df.rename(columns={
    'Year': 'year',
    'Month': 'month',
    'State': 'state',
    'Basin': 'basin',
    'Field': 'field',
    'Well': 'well',
    'Environment': 'environment',
    'Installation': 'installation',
    'Oil (m³)': 'oil_m3',
    'Condensate oil (m³)': 'condensate_oil_m3',
    'Associated petroleum gas (Mm³)': 'associated_gas_mm3',
    'Non-associated petroleum gas (Mm³)': 'non_associated_gas_mm3',
    'Water (m³)': 'water_m3',
    'Gas injection (Mm³)': 'gas_injection_mm3',
    'Secondary recovery water injection (m³)': 'secondary_water_injection_m3',
    'Wastewater injection (m³)': 'wastewater_injection_m3',
    'CO2 injection (Mm³)': 'co2_injection_mm3',
    'Nitrogen injection (Mm³)': 'nitrogen_injection_mm3',
    'Steam injection (t)': 'steam_injection_t',
    'Polymer injection (m³)': 'polymer_injection_m3',
    'Others fluids injection (m³)': 'other_fluids_injection_m3',
    'Date': 'date'
})

print(silver_df.columns.tolist())

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 22, Finished, Available, Finished, False)

['year', 'month', 'state', 'basin', 'field', 'well', 'environment', 'installation', 'oil_m3', 'condensate_oil_m3', 'associated_gas_mm3', 'non_associated_gas_mm3', 'water_m3', 'gas_injection_mm3', 'secondary_water_injection_m3', 'wastewater_injection_m3', 'co2_injection_mm3', 'nitrogen_injection_mm3', 'steam_injection_t', 'polymer_injection_m3', 'other_fluids_injection_m3', 'date']


In [21]:
print("Rows:", len(silver_df))
print("Columns:", len(silver_df.columns))
print("Duplicates:", silver_df.duplicated().sum())

print("\nData Types:")
print(silver_df.dtypes)

print("\nNegative Water Values:")
print((silver_df['water_m3'] < 0).sum())

print("\nMissing Installation:")
print(silver_df['installation'].isna().sum())

display(silver_df.head())

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 23, Finished, Available, Finished, False)

Rows: 645989
Columns: 22
Duplicates: 0

Data Types:
year                                     int64
month                                    int64
state                                   object
basin                                   object
field                                   object
well                                    object
environment                             object
installation                            object
oil_m3                                 float64
condensate_oil_m3                      float64
associated_gas_mm3                     float64
non_associated_gas_mm3                 float64
water_m3                               float64
gas_injection_mm3                      float64
secondary_water_injection_m3           float64
wastewater_injection_m3                float64
co2_injection_mm3                      float64
nitrogen_injection_mm3                 float64
steam_injection_t                      float64
polymer_injection_m3                   float64
other_fl

SynapseWidget(Synapse.DataFrame, c841738a-7e27-4eee-82b6-5771c531e450)

In [22]:
spark_silver_df = spark.createDataFrame(silver_df)

print("Rows:", spark_silver_df.count())
spark_silver_df.printSchema()

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 24, Finished, Available, Finished, False)

Rows: 645989
root
 |-- year: long (nullable = true)
 |-- month: long (nullable = true)
 |-- state: string (nullable = true)
 |-- basin: string (nullable = true)
 |-- field: string (nullable = true)
 |-- well: string (nullable = true)
 |-- environment: string (nullable = true)
 |-- installation: string (nullable = true)
 |-- oil_m3: double (nullable = true)
 |-- condensate_oil_m3: double (nullable = true)
 |-- associated_gas_mm3: double (nullable = true)
 |-- non_associated_gas_mm3: double (nullable = true)
 |-- water_m3: double (nullable = true)
 |-- gas_injection_mm3: double (nullable = true)
 |-- secondary_water_injection_m3: double (nullable = true)
 |-- wastewater_injection_m3: double (nullable = true)
 |-- co2_injection_mm3: double (nullable = true)
 |-- nitrogen_injection_mm3: double (nullable = true)
 |-- steam_injection_t: double (nullable = true)
 |-- polymer_injection_m3: double (nullable = true)
 |-- other_fluids_injection_m3: double (nullable = true)
 |-- date: timestamp (n

In [23]:
spark_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_production")

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 25, Finished, Available, Finished, False)

In [24]:
check_df = spark.table("silver_production")

print("Saved rows:", check_df.count())
display(check_df.limit(5))

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 26, Finished, Available, Finished, False)

Saved rows: 645989


SynapseWidget(Synapse.DataFrame, 8978a6b3-85f8-4e9c-af67-9594ee36ddba)

In [25]:
dim_date = silver_df[
    ['date', 'year', 'month']
].drop_duplicates().copy()

dim_date = dim_date.sort_values('date').reset_index(drop=True)

dim_date['date_key'] = (
    dim_date['year'] * 100 + dim_date['month']
)

dim_date = dim_date[
    ['date_key', 'date', 'year', 'month']
]

display(dim_date.head(15))

print("Rows:", len(dim_date))

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dc47db46-c9b1-433c-93f3-1997e99ef7aa)

Rows: 790


In [26]:
well_check = (
    silver_df.groupby('well')
    .agg(
        environment_count=('environment', 'nunique'),
        installation_count=('installation', 'nunique')
    )
)

print("Unique wells:", silver_df['well'].nunique())

print(
    "Wells with multiple environments:",
    (well_check['environment_count'] > 1).sum()
)

print(
    "Wells with multiple installations:",
    (well_check['installation_count'] > 1).sum()
)

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 28, Finished, Available, Finished, False)

Unique wells: 16369
Wells with multiple environments: 0
Wells with multiple installations: 5508


In [27]:
dim_well = silver_df[
    ['well', 'environment']
].drop_duplicates().copy()

dim_well = dim_well.sort_values('well').reset_index(drop=True)

dim_well['well_key'] = dim_well.index + 1

dim_well = dim_well[
    ['well_key', 'well', 'environment']
]

display(dim_well.head())

print("Rows:", len(dim_well))

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c109fc8a-47f3-4477-a687-ae01863766fe)

Rows: 16369


In [28]:
field_check = (
    silver_df.groupby('field')
    .agg(
        basin_count=('basin', 'nunique'),
        state_count=('state', 'nunique')
    )
)

print("Unique fields:", silver_df['field'].nunique())

print(
    "Fields with multiple basins:",
    (field_check['basin_count'] > 1).sum()
)

print(
    "Fields with multiple states:",
    (field_check['state_count'] > 1).sum()
)

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 30, Finished, Available, Finished, False)

Unique fields: 415
Fields with multiple basins: 3
Fields with multiple states: 3


In [29]:
problem_fields = field_check[
    (field_check['basin_count'] > 1) |
    (field_check['state_count'] > 1)
]

display(problem_fields)

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8c2871c2-2dea-4dc3-b7da-8bfa382f3c1a)

In [30]:
problem_field_names = problem_fields.index.tolist()

display(
    silver_df[
        silver_df['field'].isin(problem_field_names)
    ][['field', 'basin', 'state']]
    .drop_duplicates()
    .sort_values(['field', 'basin', 'state'])
)

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3442202d-2f3f-4d22-84ae-a607e3734ce6)

In [31]:
dim_field = silver_df[
    ['field', 'basin', 'state']
].drop_duplicates().copy()

dim_field = dim_field.sort_values(
    ['state', 'basin', 'field']
).reset_index(drop=True)

dim_field['field_key'] = dim_field.index + 1

dim_field = dim_field[
    ['field_key', 'field', 'basin', 'state']
]

display(dim_field.head())

print("Rows:", len(dim_field))

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 33, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f7b5302e-369e-488b-8a3f-2c68c1ac1801)

Rows: 418


In [32]:
fact_production = silver_df.merge(
    dim_date[['date', 'date_key']],
    on='date',
    how='left'
)

print(fact_production[['date', 'date_key']].head())

print(
    "Missing date_key:",
    fact_production['date_key'].isna().sum()
)

print(
    "Rows:",
    len(fact_production)
)

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 34, Finished, Available, Finished, False)

        date  date_key
0 2019-04-01    201904
1 2019-04-01    201904
2 2019-04-01    201904
3 2019-04-01    201904
4 2019-04-01    201904
Missing date_key: 0
Rows: 645989


In [33]:
fact_production = fact_production.merge(
    dim_well[['well', 'environment', 'well_key']],
    on=['well', 'environment'],
    how='left'
)

print(
    fact_production[['well', 'environment', 'well_key']].head()
)

print(
    "Missing well_key:",
    fact_production['well_key'].isna().sum()
)

print(
    "Rows:",
    len(fact_production)
)

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 35, Finished, Available, Finished, False)

             well environment  well_key
0  1-BRSA-313D-AL        Land        41
1  1-BRSA-513D-AL        Land        63
2    7-ANB-10D-AL        Land      1408
3     7-ANB-11-AL        Land      1409
4    7-ANB-12D-AL        Land      1410
Missing well_key: 0
Rows: 645989


In [34]:
fact_production = fact_production.merge(
    dim_field[['field', 'basin', 'state', 'field_key']],
    on=['field', 'basin', 'state'],
    how='left'
)

print(
    fact_production[
        ['field', 'basin', 'state', 'field_key']
    ].head()
)

print(
    "Missing field_key:",
    fact_production['field_key'].isna().sum()
)

print(
    "Rows:",
    len(fact_production)
)

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 36, Finished, Available, Finished, False)

    field    basin state  field_key
0  ANAMBÉ  Alagoas    AL          1
1  ANAMBÉ  Alagoas    AL          1
2  ANAMBÉ  Alagoas    AL          1
3  ANAMBÉ  Alagoas    AL          1
4  ANAMBÉ  Alagoas    AL          1
Missing field_key: 0
Rows: 645989


In [35]:
print(
    fact_production[['field_key']].head()
)

print(
    "Missing field_key:",
    fact_production['field_key'].isna().sum()
)

print(
    "Rows:",
    len(fact_production)
)

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 37, Finished, Available, Finished, False)

   field_key
0          1
1          1
2          1
3          1
4          1
Missing field_key: 0
Rows: 645989


In [36]:
print(fact_production.columns.tolist())

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 38, Finished, Available, Finished, False)

['year', 'month', 'state', 'basin', 'field', 'well', 'environment', 'installation', 'oil_m3', 'condensate_oil_m3', 'associated_gas_mm3', 'non_associated_gas_mm3', 'water_m3', 'gas_injection_mm3', 'secondary_water_injection_m3', 'wastewater_injection_m3', 'co2_injection_mm3', 'nitrogen_injection_mm3', 'steam_injection_t', 'polymer_injection_m3', 'other_fluids_injection_m3', 'date', 'date_key', 'well_key', 'field_key']


In [37]:
fact_production = fact_production[
    [
        'date_key',
        'well_key',
        'field_key',
        'installation',
        'oil_m3',
        'condensate_oil_m3',
        'associated_gas_mm3',
        'non_associated_gas_mm3',
        'water_m3',
        'gas_injection_mm3',
        'secondary_water_injection_m3',
        'wastewater_injection_m3',
        'co2_injection_mm3',
        'steam_injection_t',
        'polymer_injection_m3'
    ]
].copy()

print("Rows:", len(fact_production))
print("Columns:", len(fact_production.columns))
display(fact_production.head())

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 39, Finished, Available, Finished, False)

Rows: 645989
Columns: 15


SynapseWidget(Synapse.DataFrame, 12e375a4-8708-42c4-8aaf-be8555b583e4)

In [38]:
print("Missing date_key:", fact_production['date_key'].isna().sum())
print("Missing well_key:", fact_production['well_key'].isna().sum())
print("Missing field_key:", fact_production['field_key'].isna().sum())

print("Fact rows:", len(fact_production))
print("Dim date rows:", len(dim_date))
print("Dim well rows:", len(dim_well))
print("Dim field rows:", len(dim_field))

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 40, Finished, Available, Finished, False)

Missing date_key: 0
Missing well_key: 0
Missing field_key: 0
Fact rows: 645989
Dim date rows: 790
Dim well rows: 16369
Dim field rows: 418


In [39]:
spark_dim_date = spark.createDataFrame(dim_date)
spark_dim_well = spark.createDataFrame(dim_well)
spark_dim_field = spark.createDataFrame(dim_field)
spark_fact_production = spark.createDataFrame(fact_production)

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 41, Finished, Available, Finished, False)

In [40]:
spark_dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dim_date")

spark_dim_well.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dim_well")

spark_dim_field.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dim_field")

spark_fact_production.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_fact_production")

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 42, Finished, Available, Finished, False)

In [41]:
print("gold_dim_date:", spark.table("gold_dim_date").count())
print("gold_dim_well:", spark.table("gold_dim_well").count())
print("gold_dim_field:", spark.table("gold_dim_field").count())
print("gold_fact_production:", spark.table("gold_fact_production").count())

StatementMeta(, 0a95f8ae-ff4f-49b7-8b89-0395b5849dee, 43, Finished, Available, Finished, False)

gold_dim_date: 790
gold_dim_well: 16369
gold_dim_field: 418
gold_fact_production: 645989


In [1]:
from pyspark.sql import functions as F

check_2020 = (
    spark.table("gold_fact_production")
    .join(
        spark.table("gold_dim_date"),
        on="date_key",
        how="left"
    )
    .filter(F.col("year").isin(2019, 2020))
    .groupBy("year", "month")
    .agg(
        F.count("*").alias("rows"),
        F.sum("oil_m3").alias("total_oil_m3"),
        F.sum("water_m3").alias("total_water_m3"),
        F.sum("gas_injection_mm3").alias("gas_injection_mm3"),
        F.sum("secondary_water_injection_m3").alias("water_injection_m3")
    )
    .orderBy("year", "month")
)

display(check_2020)

StatementMeta(, 0e11f4e9-7876-4b0a-ad72-cf948c72b4d2, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c39f6c77-5be1-4854-8727-79a469fcdb15)

In [2]:
from pyspark.sql import functions as F

spark.table("gold_fact_production") \
    .join(spark.table("gold_dim_date"), "date_key") \
    .filter((F.col("year") == 2020)) \
    .groupBy("month") \
    .agg(
        F.count("*").alias("rows"),
        F.sum("oil_m3").alias("oil_m3")
    ) \
    .orderBy("month") \
    .show()

StatementMeta(, 0e11f4e9-7876-4b0a-ad72-cf948c72b4d2, 4, Finished, Available, Finished, False)

+-----+----+--------------------+
|month|rows|              oil_m3|
+-----+----+--------------------+
|    1|8633|1.5340192358270008E7|
|    2|8649|1.3481399134850008E7|
|    3|8961|1.4420113344619997E7|
|    4|8868| 1.384759966411999E7|
|    5|8254|1.3373590397880012E7|
|    6|8361|    1.411951049249E7|
|    7|8244|1.4931735840230012E7|
|    8|6770|  233709.36237000022|
|    9|6096|  170155.24337999953|
+-----+----+--------------------+



In [3]:
from pyspark.sql import functions as F

check_oil = (
    spark.table("gold_fact_production")
    .join(spark.table("gold_dim_date"), "date_key")
    .filter(F.col("year") == 2020)
    .groupBy("month")
    .agg(
        F.count("*").alias("total_rows"),
        F.count("oil_m3").alias("oil_reported_rows"),
        F.sum("oil_m3").alias("total_oil_m3"),
        F.avg("oil_m3").alias("avg_oil_m3"),
        F.max("oil_m3").alias("max_oil_m3")
    )
    .orderBy("month")
)

display(check_oil)

StatementMeta(, 0e11f4e9-7876-4b0a-ad72-cf948c72b4d2, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 56b65e40-84c1-4af1-b7a5-cc48e3f2e58d)

In [4]:
from pyspark.sql import functions as F

check_2020_metrics = (
    spark.table("gold_fact_production")
    .join(spark.table("gold_dim_date"), "date_key")
    .filter(F.col("year") == 2020)
    .groupBy("month")
    .agg(
        F.sum("oil_m3").alias("oil_m3"),
        F.sum("water_m3").alias("water_m3"),
        F.sum("associated_gas_mm3").alias("associated_gas_mm3"),
        F.sum("gas_injection_mm3").alias("gas_injection_mm3"),
        F.sum("secondary_water_injection_m3").alias("water_injection_m3")
    )
    .orderBy("month")
)

display(check_2020_metrics)

StatementMeta(, 0e11f4e9-7876-4b0a-ad72-cf948c72b4d2, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8fa883d9-2ea7-44ca-ac8d-2455c03777e2)